# KM Momentum — 6M — Equal Weight

One signal, one formation horizon and one maintenance method. The experiment contains nine cells: rebalance every 1, 3 or 6 months × target N=12, 24 or 50.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'src'))
from momentum_india.notebook_views import ResearchNotebook

research = ResearchNotebook('km_momentum', '6M', 'equal_weight')

## 1. Signal and portfolio rule

Quality/liquidity universe → Close ≥ 1.03 × EMA(100) and bullish Supertrend(10,3) → raw-momentum ranking → highest N. There is no secondary proximity or risk ranking. A daily close below EMA(100), or a bearish Supertrend, triggers a sale at the next actual observed open. The proceeds earn the cash-sleeve return until the next scheduled rebalance.

### Technical definition

EMA(100) begins at the mean of the first 100 observed closes and thereafter uses α=2/101. ATR(10) begins at the mean of the first ten true ranges having a prior close, then uses Wilder's α=1/10 smoothing. Supertrend bands are HL2 ± 3×ATR; the prior bands govern direction changes and the active band cannot retreat while direction is unchanged. Direction starts bullish at the first valid ATR bar. Missing observations do not create synthetic indicator bars.

These initialization rules make the calculation reproducible. They do not claim exact equality with an unidentified historical charting-package version. All indicators use split/bonus-adjusted OHLC. The 3% buffer applies only to entry; the EMA exit has no 3% buffer.

Each selected stock targets 1/N of pre-trade equity at a scheduled rebalance. If K<N qualify, target cash is (N−K)/N before charges; eligible stocks are not enlarged to 1/K. Sales precede purchases, which are reduced proportionally if charges or unfilled sales restrict available cash. Actual weights can therefore differ slightly from targets. Membership is rebuilt from current ranks without a retention buffer.

## 2. Universe → ranking → actual portfolio

The example uses the latest monthly N=24 signal and exposes the ranking inputs and actual target weights.

In [2]:
research.snapshot()

Symbol,Formation price return,MDTV (INR)
NSE:YASHO-EQ,216.2%,"37,220,195.50"
NSE:CUPID-EQ,187.6%,"2,261,224,535.95"
NSE:DIACABS-EQ,164.4%,"490,913,783.79"
NSE:CONFIPET-EQ,145.8%,"129,439,974.50"
NSE:KSHINTL-EQ,142.6%,"194,129,104.07"
NSE:SETL-EQ,138.1%,"42,154,727.49"
NSE:GRWRHITECH-EQ,137.4%,"309,324,106.35"
NSE:RPTECH-EQ,132.2%,"78,147,051.55"
NSE:SHADOWFAX-EQ,127.8%,"320,801,875.19"
NSE:SAKAR-EQ,126.3%,"72,322,117.10"


Symbol,Leg,Actual weight,Current selection,Execution status,Formation price return,MDTV (INR)
NSE:YASHO-EQ,long,3.6%,True,selected,216.2%,"37,220,195.50"
NSE:CUPID-EQ,long,2.3%,True,selected,187.6%,"2,261,224,535.95"
NSE:DIACABS-EQ,long,2.3%,True,selected,164.4%,"490,913,783.79"
NSE:CONFIPET-EQ,long,3.9%,True,selected,145.8%,"129,439,974.50"
NSE:KSHINTL-EQ,long,3.2%,True,selected,142.6%,"194,129,104.07"
NSE:SETL-EQ,long,2.3%,True,selected,138.1%,"42,154,727.49"
NSE:GRWRHITECH-EQ,long,3.3%,True,selected,137.4%,"309,324,106.35"
NSE:RPTECH-EQ,long,3.3%,True,selected,132.2%,"78,147,051.55"
NSE:SHADOWFAX-EQ,long,2.3%,True,selected,127.8%,"320,801,875.19"
NSE:SAKAR-EQ,long,3.9%,True,selected,126.3%,"72,322,117.10"


Symbol,Formation start,Start adjusted close,Signal close date,End adjusted close,Formation price return
NSE:YASHO-EQ,2026-01-30,"1,220.30",2026-07-31,"3,858.70",216.2%


## 3. Return layers across all nine cells

Raw is before trading charges. After-cost gross/pre-tax deducts modeled trading charges. The long-only post-tax overlay additionally applies the annual equity-gains ledger. The academic reference instead compares raw and borrowing-adjusted layers.

In [3]:
research.layers_bridge()

Rebalance,First date,Last date,Sessions
1M,2007-05-03,2026-08-28,4771
3M,2007-07-02,2026-08-28,4729
6M,2007-07-02,2026-08-28,4729


Rebalance,N,Raw,After costs,Post-tax overlay
1M,12,19.0%,17.1%,15.0%
1M,24,21.8%,19.9%,17.3%
1M,50,21.8%,20.0%,17.4%
3M,12,22.8%,21.8%,19.2%
3M,24,23.5%,22.5%,19.7%
3M,50,21.5%,20.6%,18.2%
6M,12,6.2%,5.8%,5.3%
6M,24,9.3%,8.8%,8.0%
6M,50,10.0%,9.5%,8.7%


## 4. Risk-adjusted results

Sharpe uses daily excess returns relative to the liquid fund. VaR and expected shortfall are historical monthly 95% loss measures. Partial first/last months are included. Time below prior peak counts days awaiting a new all-time high—not losing days. The initial invested capital is included as the first peak.

In [4]:
research.risk_grid()

Rebalance,N,CAGR,Sharpe,Max drawdown,Monthly VaR 95%
1M,12,15.0%,0.474,-58.2%,8.9%
1M,24,17.3%,0.619,-54.2%,8.4%
1M,50,17.4%,0.668,-48.4%,7.3%
3M,12,19.2%,0.729,-34.1%,6.7%
3M,24,19.7%,0.842,-30.1%,6.0%
3M,50,18.2%,0.834,-30.0%,5.4%
6M,12,5.3%,-0.081,-28.7%,7.0%
6M,24,8.0%,0.126,-26.4%,4.8%
6M,50,8.7%,0.199,-24.6%,4.3%


Rebalance,N,Monthly ES 95%,Time below prior peak,Longest underwater (sessions)
1M,12,11.3%,94.8%,1648
1M,24,10.9%,92.2%,814
1M,50,10.0%,91.5%,832
3M,12,9.6%,93.0%,630
3M,24,8.3%,91.2%,741
3M,50,7.6%,90.5%,754
6M,12,9.5%,96.6%,809
6M,24,7.6%,94.4%,762
6M,50,6.9%,93.1%,767


### CAGR

In [5]:
research.heatmap('cagr')

### Sharpe ratio

In [6]:
research.heatmap('sharpe')

### Maximum drawdown

In [7]:
research.heatmap('maximum_drawdown')

### Monthly 95% VaR

In [8]:
research.heatmap('monthly_var_95')

## 5. Equity paths and matched risks

Each chart fixes breadth and compares rebalance frequencies. Final-layer curves and the price benchmark start visible; other return layers remain in the selectable legend. Logarithmic axes make early and late periods comparable; the bottom range slider preserves the full history. Curves display weekly observations for readability, while every statistic uses the complete daily series.

### N=12

In [9]:
research.equity(12)

Rebalance,Return layer,CAGR,Sharpe,Max drawdown,Monthly VaR 95%
1M,Raw,19.0%,0.659,-55.4%,7.8%
1M,After costs,17.1%,0.577,-56.8%,7.9%
1M,Post-tax overlay,15.0%,0.474,-58.2%,8.9%
1M,Nifty 50 price,9.7%,0.219,-59.9%,6.9%
3M,Raw,22.8%,0.920,-29.5%,6.6%
3M,After costs,21.8%,0.871,-30.5%,6.7%
3M,Post-tax overlay,19.2%,0.729,-34.1%,6.7%
3M,Nifty 50 price,9.5%,0.209,-59.9%,7.0%
6M,Raw,6.2%,-0.009,-27.8%,6.9%
6M,After costs,5.8%,-0.045,-28.3%,7.0%


Rebalance,Return layer,Monthly ES 95%,Time below prior peak,Longest underwater (sessions)
1M,Raw,10.5%,93.4%,1574
1M,After costs,10.6%,94.0%,1597
3M,Raw,9.3%,91.8%,580
3M,After costs,9.4%,92.0%,584
6M,Raw,9.4%,96.2%,639
6M,After costs,9.5%,96.3%,792
1M,Post-tax overlay,11.3%,94.8%,1648
3M,Post-tax overlay,9.6%,93.0%,630
6M,Post-tax overlay,9.5%,96.6%,809
1M,Nifty 50 price,13.1%,92.4%,1520


### N=24

In [10]:
research.equity(24)

Rebalance,Return layer,CAGR,Sharpe,Max drawdown,Monthly VaR 95%
1M,Raw,21.8%,0.845,-51.4%,8.2%
1M,After costs,19.9%,0.757,-52.6%,8.3%
1M,Post-tax overlay,17.3%,0.619,-54.2%,8.4%
1M,Nifty 50 price,9.7%,0.219,-59.9%,6.9%
3M,Raw,23.5%,1.076,-29.3%,5.5%
3M,After costs,22.5%,1.020,-30.1%,5.7%
3M,Post-tax overlay,19.7%,0.842,-30.1%,6.0%
3M,Nifty 50 price,9.5%,0.209,-59.9%,7.0%
6M,Raw,9.3%,0.241,-25.6%,4.6%
6M,After costs,8.8%,0.197,-26.2%,4.8%


Rebalance,Return layer,Monthly ES 95%,Time below prior peak,Longest underwater (sessions)
1M,Raw,10.0%,90.5%,762
1M,After costs,10.1%,91.2%,807
3M,Raw,7.9%,89.6%,727
3M,After costs,8.0%,90.0%,736
6M,Raw,7.3%,93.6%,637
6M,After costs,7.5%,93.8%,638
1M,Post-tax overlay,10.9%,92.2%,814
3M,Post-tax overlay,8.3%,91.2%,741
6M,Post-tax overlay,7.6%,94.4%,762
1M,Nifty 50 price,13.1%,92.4%,1520


### N=50

In [11]:
research.equity(50)

Rebalance,Return layer,CAGR,Sharpe,Max drawdown,Monthly VaR 95%
1M,Raw,21.8%,0.910,-45.4%,6.8%
1M,After costs,20.0%,0.818,-46.5%,6.9%
1M,Post-tax overlay,17.4%,0.668,-48.4%,7.3%
1M,Nifty 50 price,9.7%,0.219,-59.9%,6.9%
3M,Raw,21.5%,1.072,-25.2%,5.1%
3M,After costs,20.6%,1.011,-26.2%,5.2%
3M,Post-tax overlay,18.2%,0.834,-30.0%,5.4%
3M,Nifty 50 price,9.5%,0.209,-59.9%,7.0%
6M,Raw,10.0%,0.330,-24.0%,4.1%
6M,After costs,9.5%,0.280,-24.6%,4.3%


Rebalance,Return layer,Monthly ES 95%,Time below prior peak,Longest underwater (sessions)
1M,Raw,9.0%,89.3%,809
1M,After costs,9.1%,90.2%,814
3M,Raw,7.2%,88.8%,706
3M,After costs,7.4%,89.2%,727
6M,Raw,6.6%,91.8%,629
6M,After costs,6.8%,92.1%,754
1M,Post-tax overlay,10.0%,91.5%,832
3M,Post-tax overlay,7.6%,90.5%,754
6M,Post-tax overlay,6.9%,93.1%,767
1M,Nifty 50 price,13.1%,92.4%,1520


## 6. Recovery burden

The longest underwater episode is shown by its peak, trough and recovery dates. Unrecovered episodes remain explicitly open.

In [12]:
research.recovery()

Series,Peak,Trough,Recovery,Sessions,Calendar days,Episode loss
1M,2018-01-15,2019-09-19,2021-05-10,814,1210,-45.9%
3M,2010-11-11,2011-11-21,2013-11-11,741,1095,-22.3%
6M,2018-01-15,2019-08-23,2021-02-18,762,1129,-17.4%
Nifty · 1M,2008-01-08,2008-10-27,2014-03-06,1520,2248,-59.9%
Nifty · 3M,2008-01-08,2008-10-27,2014-03-06,1520,2248,-59.9%
Nifty · 6M,2008-01-08,2008-10-27,2014-03-06,1520,2248,-59.9%


## 7. Portfolio behavior

Scheduled turnover is (buy value + sell value)/(2 × pre-trade equity). Retention compares successive scheduled target name sets. Cash and total charges include the intervening daily path.

In [13]:
research.behavior()

Rebalance,N,Mean names at rebalance,Mean cash weight,Scheduled name retention
1M,12,19.42,14.6%,58.1%
1M,24,35.66,15.5%,58.0%
1M,50,69.38,16.7%,59.0%
3M,12,14.88,41.1%,29.2%
3M,24,28.31,41.7%,30.1%
3M,50,57.83,42.1%,34.5%
6M,12,13.85,60.3%,13.7%
6M,24,26.69,61.9%,12.0%
6M,50,54.10,63.0%,13.6%


Rebalance,N,Mean scheduled turnover,All trading charges (INR),Days with stop sales
1M,12,32.8%,"19,858,775.35",829
1M,24,33.2%,"28,715,909.40",1277
1M,50,31.4%,"27,380,371.09",1929
3M,12,47.0%,"19,616,466.79",575
3M,24,47.3%,"21,347,590.66",927
3M,50,44.2%,"16,182,906.16",1490
6M,12,45.8%,"1,635,595.63",349
6M,24,47.6%,"2,185,117.73",603
6M,50,47.8%,"2,429,420.19",990


## 8. Market-state attribution

This is an observation, not an extra strategy filter. Prior-close Nifty 50 versus SMA(200) defines up/down; 63-session volatility versus its expanding historical median defines high/low volatility. The representative monthly N=24 path is shown with shaded states.

In [14]:
research.regimes()

Market state,Sessions,Mean daily return,Daily volatility,Positive days
Down / High volatility,748,0.02%,0.92%,54.95%
Down / Low volatility,609,-0.05%,1.06%,52.55%
Up / High volatility,720,0.23%,1.32%,62.22%
Up / Low volatility,2694,0.07%,1.08%,57.05%


## 9. Complete portfolio and trade evidence

Separate CSV files retain all scheduled portfolios, actual trades and risk layers for this exact signal/lookback/maintenance combination.

In [15]:
research.portfolio_exports()

Rebalance,N,First rebalance,Last rebalance,Rebalance dates,Holding rows
1M,12,2007-05-03,2026-08-03,232,4505
1M,24,2007-05-03,2026-08-03,232,8273
1M,50,2007-05-03,2026-08-03,232,16096
3M,12,2007-07-02,2026-07-01,77,1146
3M,24,2007-07-02,2026-07-01,77,2180
3M,50,2007-07-02,2026-07-01,77,4453
6M,12,2007-07-02,2026-07-01,39,540
6M,24,2007-07-02,2026-07-01,39,1041
6M,50,2007-07-02,2026-07-01,39,2110


## Findings

In [16]:
research.conclusion()

Matched reference: [Raw Momentum — 6M — Equal Weight](02_raw_momentum_6m_equal_weight.ipynb). The comparison holds formation, maintenance, frequency and N fixed.